In [ ]:
# Check our directory just in case
%cd /content/

# Clone TTPython and Dependencies
%rm -r ticktalkpython
!git clone https://bitbucket.org/ccsg-res/ticktalkpython.git
!cd ticktalkpython && git checkout cli
!cd ticktalkpython && pip install -r requirements.txt

# These are to store code written in this notebook
import inspect, types
def __get_wrapped_source(decorated):
    closure = (c.cell_contents for c in decorated.__closure__)
    extracted = next((c for c in closure if isinstance(c, types.FunctionType)), None)
    return inspect.getsource(extracted)
def __get_source(funcs):
    f_set = set(funcs)
    values = globals().values()
    is_func = lambda x: type(x) == types.FunctionType
    name = lambda x: x.__name__
    ordered = [f for f in values if is_func(f) and f in funcs]
    source = list(map(__get_wrapped_source, ordered))
    return '\n'.join(source)
def tt_store(output, headers, funcs):
    funcs_source = __get_source(funcs)
    headers_source = '\n'.join(headers)
    source = f"{headers_source}\n{funcs_source}"
    with open(output, 'w') as f:
        f.write(source)

# For TTPython access
import sys
sys.path.append('tt/')

# We are done!
print("\nTTPython setup complete, proceed to next block.")

#**Graph Define**

Define the dataflow graph

In [ ]:
# Check our directory just in case
%cd /content/ticktalkpython/

from logging import root
from SQ import STREAMify, GRAPHify
from Clock import TTClock
from Instructions import *

@STREAMify #streamify is meant for generating sampled data streams
def sinusoid_sampler(A, f, phi):
    from math import sin, pi
    global sq_state
    if sq_state.get('count', None) == None:
        sq_state['count'] = 1

    sample = A * sin(sq_state['count'] * 2*f/pi + phi)
    sq_state['count'] += 1

    return sample

@SQify
def movingAverage(new_input):
    global sq_state
    count = sq_state.get('count', 0)
    average = (sq_state.get('average', 0) * count + new_input) / (count+1)

    sq_state['count'] = count + 1
    sq_state['average'] = average

    return average

@GRAPHify
def streamify_test(trigger):
    A_1 = 1
    A_2 = 2
    f_1 = 0.25
    f_2 = 0.25
    phi_1 = 0
    phi_2 = 0

    with TTClock.root() as root_clock:
        # collect a timestamp from a clock; needs a trigger whose arrival will make the timestamp be taken. This is for setting the start-tick of the STREAMify's periodic firing rule
        start_time = READ_TTCLOCK(trigger, TTClock=root_clock)
        N = 30
        # Setup the stop-tick of the STREAMify's firing rule
        stop_time = start_time + (1000000 * N) # sample for N seconds

        # create a sampling interval by copying the start and stop tick from token values to the token time interval
        sampling_time = VALUES_TO_TTTIME(start_time, stop_time)

        # copy the sampling interval to the input values to the STREAMify node; these input values will be treated as sticky tokens, and define the duration over which STREAMify'd nodes must run
        A1_sample = COPY_TTTIME(A_1, sampling_time)
        A2_sample = COPY_TTTIME(A_2, sampling_time)

        # do the sampling with streamify'd SQs. Only one of the inputs needs the special sampling time interval (but it wouldn't hurt if all did) because the other const values have infinite timestamps
        sine_1 = sinusoid_sampler(A1_sample, f_1, phi_1, TTClock=root_clock, TTPeriod=500000, TTPhase=0, TTDataIntervalWidth=100000)
        sine_2 = sinusoid_sampler(A2_sample, f_2, phi_2, TTClock=root_clock, TTPeriod=500000, TTPhase=0, TTDataIntervalWidth=100000)

        # do some operations on the streams at runtime. Multiple streams will have their values synchronized by searching for intersections/overlaps in their time-intervals
        output = sine_1 + sine_2
        y = movingAverage(sine_1)
        return output

Normally, just save the above in a python file. Since this isn't possible in this notebook, use the hacky solution below.

In [ ]:
# For Google Colab to store above code as file
headers = [
    "from logging import root",
    "from SQ import STREAMify, GRAPHify",
    "from Clock import TTClock",
    "from Instructions import *",
]
funcs = [sinusoid_sampler, movingAverage, streamify_test]
tt_store('graph.py', headers, funcs)

Now, compile the file into graph (.pickle) file.

In [ ]:
# Check our directory just in case
%cd /content/ticktalkpython/

# Here are some compilation options
!python compile.py -help

In [ ]:
!python compile.py graph.py

#**Simulation**

Run a simulation over the dataflow graph

In [ ]:
# Check our directory just in case
%cd /content/ticktalkpython/

# Here are some simulation options
!python simulate.py -help

In [ ]:
!python simulate.py graph.pickle
print("\nCompleted simulation")

The simulation data is stored in `output.log`. Now to visualize it.

In [ ]:
# Check our directory just in case
%cd /content/ticktalkpython/

# Here are some visualization options
!python visualize.py -help

In [ ]:
!python visualize.py output.log -list

In [ ]:
%matplotlib inline
# %run is used for the purposes of this notebook (use python or ./)
%run visualize.py output.log -name graph ADD-16

In [ ]:
%matplotlib inline
# %run is used for the purposes of this notebook (use python or ./)
%run visualize.py output.log -name graph movingAverage-17